# 07 · Freeze the dataset and run sanity checks

### What this notebook does
1. Writes `processed/dataset_info.json` (sizes, padding index, sequence length, subset).
2. Writes **`processed/rec_eval.py`**, the shared data-loading and evaluation module every model must import, so
   inputs, history masking and metrics are identical for Models 0–4.
3. Sanity-checks the evaluation code with known answers:
   - an **oracle** that always ranks the true item first must score exactly 1.0;
   - a **random** model must score close to its theoretical value (≈ K / catalog size);
   - a **popularity** baseline must beat random.
4. Checks the SASRec training arrays and the cold-start and client files load correctly.
5. Writes the **manifest** (every file with SHA-256 checksum and size) and a **data card** for the report.

### Why
If the evaluation code is wrong, every model comparison is wrong. Testing it on cases with known answers *before*
training Model 0 is the cheapest way to catch bugs. The popularity numbers also become the minimum bar SASRec must
clearly beat. The manifest is the "freeze": after this, nobody changes the data.

**Inputs:** everything in `processed/` · **Outputs:** `processed/dataset_info.json`, `processed/rec_eval.py`,
`processed/manifest.json`, `processed/DATA_CARD.md`, `reports/sanity_baselines.csv`

In [ ]:
# ---------------- Shared setup (identical in every notebook) ----------------
# All notebooks read and write inside one project folder (BASE).
# Colab: BASE lives in your Google Drive so outputs survive runtime restarts.
# Local Jupyter: BASE is ./298A_recsys next to the notebooks.
# Set the environment variable RECSYS_BASE to use any other folder.
import os, sys, json, re, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules
if os.environ.get('RECSYS_BASE'):
    BASE = Path(os.environ['RECSYS_BASE'])
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/298A_recsys')
else:
    BASE = Path('./298A_recsys').resolve()

RAW, INTERIM, PROCESSED = BASE / 'raw', BASE / 'interim', BASE / 'processed'
REPORTS, FIGS = BASE / 'reports', BASE / 'reports' / 'figures'
# Large downloads go to fast local disk in Colab (not Drive); re-downloaded if the runtime resets.
DOWNLOADS = Path(os.environ.get('RECSYS_DOWNLOAD_DIR', '/content/downloads' if IN_COLAB else str(BASE / 'downloads')))
for d in [RAW, INTERIM, PROCESSED, REPORTS, FIGS, DOWNLOADS]:
    d.mkdir(parents=True, exist_ok=True)

# Plot style: one hue per single-series chart, light recessive grid, no chart junk.
PRIMARY, ACCENT, MUTED = '#2a78d6', '#eb6834', '#8a8984'
plt.rcParams.update({
    'figure.figsize': (8, 4), 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'axes.prop_cycle': matplotlib.cycler(color=[PRIMARY, ACCENT, '#1baf7a', '#eda100']),
})

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGS / f'{name}.png', bbox_inches='tight')
    plt.show()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_config():
    cfg_path = BASE / 'config.json'
    if not cfg_path.exists():
        raise FileNotFoundError('config.json not found - run 01_data_acquisition.ipynb first.')
    return json.loads(cfg_path.read_text())

print('Project folder:', BASE)

In [ ]:
CFG = load_config()
index_info = json.loads((INTERIM / 'index_info.json').read_text())
required = ['interactions.parquet', 'sequences.parquet', 'items.parquet', 'item_text.parquet', 'item_map.parquet',
            'user_map.parquet', 'leakage_report.json', 'eval/eval_protocol.json', 'eval/neg_val.npy',
            'eval/neg_test.npy', 'eval/eval_cases.parquet', 'groups/user_groups.parquet', 'groups/item_groups.parquet',
            'cold_start/cold_items.npy', 'cold_start/cold_sequences.parquet', 'clients/clients.parquet',
            'interaction_side_info.parquet']
missing = [f for f in required if not (PROCESSED / f).exists()]
assert not missing, f'Missing files (re-run earlier notebooks): {missing}'
leak = json.loads((PROCESSED / 'leakage_report.json').read_text())
assert leak['all_passed'], 'Leakage checks in notebook 05 did not all pass.'
print('All required files present; leakage report: all checks passed.')

## 1. Dataset info

In [ ]:
dataset_info = {
    'dataset': CFG['dataset'],
    'n_users': index_info['n_users'],
    'n_items': index_info['n_items'],
    'n_interactions': index_info['n_interactions'],
    'padding_idx': 0,
    'item_idx_range': [1, index_info['n_items']],
    'max_seq_len': CFG['max_seq_len'],
    'kcore': CFG['kcore'],
    'split': 'chronological leave-one-out (last = test, second-last = val)',
    'subset': index_info['subset'],
    'seed': CFG['seed'],
}
save_json(dataset_info, PROCESSED / 'dataset_info.json')
dataset_info

## 2. Shared evaluation module
The module below is written to `processed/rec_eval.py`. Model notebooks import it with
`sys.path.insert(0, str(PROCESSED)); import rec_eval as R`.

In [ ]:
# Source of the shared evaluation module. It is written to processed/rec_eval.py and imported below.
MODULE_SRC = r'''"""
rec_eval.py - shared data loading and evaluation for Models 0-4.
Generated by 07_freeze_and_sanity_check.ipynb. Do not edit by hand: every model must use this exact file so that
inputs, masking and metrics are identical across the baseline, LLM-enhanced, federated, on-device and DP variants.

Usage (from any model notebook):
    import sys; sys.path.insert(0, str(PROCESSED))
    import rec_eval as R
    data = R.load_dataset(PROCESSED)
    X_tr, Y_tr = R.train_arrays(data['sequences'], data['max_seq_len'])
    res = R.evaluate(score_fn, data, split='val')            # full-catalog ranking (main protocol)
    res = R.evaluate(score_fn, data, split='test', sampled=True)  # 100 sampled negatives (secondary)

score_fn(inputs, users) -> scores
    inputs: int64 array (B, max_seq_len), left-padded with 0, most recent item last
    users : int64 array (B,) of user_idx
    scores: float array (B, n_items + 1); column 0 is padding and is ignored
"""
import json
from pathlib import Path
import numpy as np
import pandas as pd


def load_dataset(processed_dir):
    p = Path(processed_dir)
    info = json.loads((p / 'dataset_info.json').read_text())
    seq = pd.read_parquet(p / 'sequences.parquet').sort_values('user_idx').reset_index(drop=True)
    return {
        'info': info,
        'n_items': info['n_items'],
        'n_users': info['n_users'],
        'max_seq_len': info['max_seq_len'],
        'sequences': seq,
        'protocol': json.loads((p / 'eval' / 'eval_protocol.json').read_text()),
        'neg_val': np.load(p / 'eval' / 'neg_val.npy'),
        'neg_test': np.load(p / 'eval' / 'neg_test.npy'),
        'eval_cases': pd.read_parquet(p / 'eval' / 'eval_cases.parquet'),
        'item_groups': pd.read_parquet(p / 'groups' / 'item_groups.parquet'),
        'user_groups': pd.read_parquet(p / 'groups' / 'user_groups.parquet'),
    }


def left_pad(seqs, max_len):
    """Keep the last max_len items of each sequence; left-pad with 0."""
    out = np.zeros((len(seqs), max_len), dtype=np.int64)
    for i, s in enumerate(seqs):
        s = list(s)[-max_len:]
        if s:
            out[i, max_len - len(s):] = s
    return out


def train_arrays(sequences, max_len):
    """SASRec-style training arrays: input = items[:-1], target = items[1:] (next item at every position)."""
    inputs, targets = [], []
    for s in sequences['train_items']:
        s = list(s)[-(max_len + 1):]
        inputs.append(s[:-1])
        targets.append(s[1:])
    return left_pad(inputs, max_len), left_pad(targets, max_len)


def eval_arrays(sequences, split, max_len):
    """Inputs, targets and full histories (for masking) for 'val' or 'test'."""
    if split == 'val':
        hists = [list(t) for t in sequences['train_items']]
        targets = sequences['val_item'].to_numpy()
    elif split == 'test':
        hists = [list(t) + [int(v)] for t, v in zip(sequences['train_items'], sequences['val_item'])]
        targets = sequences['test_item'].to_numpy()
    else:
        raise ValueError("split must be 'val' or 'test'")
    return left_pad(hists, max_len), targets.astype(np.int64), hists


def _tie_aware_rank(target_score, cand_scores):
    """Expected rank under random tie-breaking: 1 + #greater + #equal_others / 2."""
    greater = (cand_scores > target_score[:, None]).sum(1)
    equal = (cand_scores == target_score[:, None]).sum(1) - 1
    return 1.0 + greater + equal / 2.0


def metrics_from_ranks(ranks, ks=(5, 10, 20)):
    ranks = np.asarray(ranks, dtype=float)
    out = {}
    for k in ks:
        hit = ranks <= k
        out[f'Recall@{k}'] = float(hit.mean())
        out[f'HitRate@{k}'] = float(hit.mean())          # identical to Recall@K with one target per user
        out[f'NDCG@{k}'] = float(np.where(hit, 1.0 / np.log2(ranks + 1.0), 0.0).mean())
    out['MRR'] = float((1.0 / ranks).mean())
    return out


def evaluate(score_fn, data, split='val', sampled=False, ks=None, batch_size=256, users=None, return_topk=True):
    """Rank the true next item. Main protocol: full catalog with history masking. Secondary: sampled negatives."""
    ks = tuple(ks or data['protocol']['ks'])
    seq = data['sequences'] if users is None else data['sequences'][data['sequences'].user_idx.isin(users)]
    X, y, hists = eval_arrays(seq, split, data['max_seq_len'])
    u = seq['user_idx'].to_numpy().astype(np.int64)
    rows = seq.index.to_numpy()
    negs = data['neg_val'] if split == 'val' else data['neg_test']
    kmax = max(ks)
    ranks = np.zeros(len(y))
    topk = np.zeros((len(y), kmax), dtype=np.int64) if (return_topk and not sampled) else None
    for b in range(0, len(y), batch_size):
        sl = slice(b, b + batch_size)
        scores = np.asarray(score_fn(X[sl], u[sl]), dtype=np.float64)
        if scores.shape[1] != data['n_items'] + 1:
            raise ValueError(f"score_fn must return n_items + 1 = {data['n_items'] + 1} columns")
        tgt = y[sl]
        if sampled:
            cand = np.concatenate([tgt[:, None], negs[rows[sl]]], axis=1)
            cs = np.take_along_axis(scores, cand, axis=1)
            ranks[sl] = _tie_aware_rank(cs[:, 0], cs)
        else:
            scores = scores.copy()
            scores[:, 0] = -np.inf
            for i, h in enumerate(hists[sl]):
                scores[i, h] = -np.inf                        # never recommend what the user already has
            ts = scores[np.arange(len(tgt)), tgt]
            ranks[sl] = _tie_aware_rank(ts, scores)
            if topk is not None:
                part = np.argpartition(-scores, kmax, axis=1)[:, :kmax]
                order = np.argsort(-np.take_along_axis(scores, part, axis=1), axis=1)
                topk[sl] = np.take_along_axis(part, order, axis=1)
    res = {'split': split, 'protocol': 'sampled' if sampled else 'full', 'n_cases': int(len(y)),
           'metrics': metrics_from_ranks(ranks, ks), 'ranks': ranks, 'user_idx': u}
    if topk is not None:
        res['topk'] = topk
        res['metrics'].update({f'Coverage@{k}': coverage(topk[:, :k], data['n_items']) for k in ks})
    return res


def coverage(topk, n_items):
    """Share of the catalog that appears in at least one user's top-K list."""
    return float(len(np.unique(topk)) / n_items)


def category_diversity(topk, item_category):
    """Mean share of distinct categories in each top-K list (1.0 = all different). item_category indexed by item_idx."""
    cats = np.asarray(item_category, dtype=object)[topk]
    return float(np.mean([len(set(r)) / len(r) for r in cats]))


def group_breakdown(res, data, ks=None):
    """Metrics per user group (history length) and per target-item group (head / mid / tail)."""
    ks = tuple(ks or data['protocol']['ks'])
    cases = data['eval_cases']
    cases = cases[cases.split == res['split']].set_index('user_idx')
    df = pd.DataFrame({'user_idx': res['user_idx'], 'rank': res['ranks']}).join(cases, on='user_idx')
    rows = []
    for col in ['user_group', 'target_item_group']:
        for g, d in df.groupby(col):
            m = metrics_from_ranks(d['rank'].to_numpy(), ks)
            rows.append({'grouping': col, 'group': g, 'n_cases': len(d), **m})
    return pd.DataFrame(rows)
'''
(PROCESSED / 'rec_eval.py').write_text(MODULE_SRC)
print('Wrote', PROCESSED / 'rec_eval.py')

In [ ]:
import importlib
sys.path.insert(0, str(PROCESSED))
import rec_eval as R
importlib.reload(R)
data = R.load_dataset(PROCESSED)
N_ITEMS, KS = data['n_items'], tuple(CFG['eval_ks'])
seq = data['sequences']
print(f"Loaded {data['n_users']:,} users, {N_ITEMS:,} items, max_seq_len={data['max_seq_len']}")

## 3. Training and evaluation arrays

In [ ]:
X_tr, Y_tr = R.train_arrays(seq, data['max_seq_len'])
X_val, y_val, h_val = R.eval_arrays(seq, 'val', data['max_seq_len'])
X_te, y_te, h_te = R.eval_arrays(seq, 'test', data['max_seq_len'])
print('train inputs', X_tr.shape, '| train targets', Y_tr.shape, '| val inputs', X_val.shape, '| test inputs', X_te.shape)

assert X_tr.min() >= 0 and X_tr.max() <= N_ITEMS
assert ((X_tr == 0) == (Y_tr == 0)).all(), 'padding positions of inputs and targets must align'
both = (X_tr[:, :-1] > 0) & (X_tr[:, 1:] > 0)
assert (Y_tr[:, :-1][both] == X_tr[:, 1:][both]).all(), 'target at t must equal input at t+1'
assert (X_te[:, -1] == seq.val_item.values).all(), 'the last test input must be the validation item'
print('Example user 0 | last 8 training inputs:', X_tr[0, -8:], '| targets:', Y_tr[0, -8:])
print('               | val target:', y_val[0], '| test input ends with', X_te[0, -3:], '-> test target', y_te[0])

## 4. Evaluation code sanity checks
### 4a. Oracle: must score exactly 1.0

In [ ]:
target_by_user = {'val': dict(zip(seq.user_idx, seq.val_item)), 'test': dict(zip(seq.user_idx, seq.test_item))}

def oracle_fn(split):
    def f(inputs, users):
        s = np.zeros((len(users), N_ITEMS + 1), dtype=np.float32)
        s[np.arange(len(users)), [target_by_user[split][u] for u in users]] = 1.0
        return s
    return f

for split in ['val', 'test']:
    for sampled in [False, True]:
        m = R.evaluate(oracle_fn(split), data, split=split, sampled=sampled)['metrics']
        assert abs(m['NDCG@10'] - 1) < 1e-9 and abs(m['MRR'] - 1) < 1e-9, (split, sampled, m)
print('PASS oracle scores 1.0 on val/test, full and sampled')

### 4b. Random and popularity baselines
- **Random:** expected Recall@K ≈ K / (number of unmasked items) for full ranking, and K / 101 for sampled ranking.
- **Popularity:** score = training count of each item (training split only). This is the floor every model,
  including SASRec, must clearly beat.

In [ ]:
rng = np.random.default_rng(CFG['seed'])
random_fn = lambda inputs, users: rng.random((len(users), N_ITEMS + 1), dtype=np.float32)
pop = np.zeros(N_ITEMS + 1, dtype=np.float32)
ig = data['item_groups']
pop[ig.item_idx.values] = ig.train_count.values
pop_fn = lambda inputs, users: np.broadcast_to(pop, (len(users), N_ITEMS + 1))

results, breakdowns = [], []
for name, fn in [('random', random_fn), ('popularity', pop_fn)]:
    for split in ['val', 'test']:
        for sampled in [False, True]:
            res = R.evaluate(fn, data, split=split, sampled=sampled)
            results.append({'model': name, 'split': split, 'protocol': res['protocol'], **res['metrics']})
            if name == 'popularity' and split == 'test' and not sampled:
                pop_test_res = res
baselines = pd.DataFrame(results)
baselines.to_csv(REPORTS / 'sanity_baselines.csv', index=False)
show_cols = ['model', 'split', 'protocol'] + [f'Recall@{k}' for k in KS] + [f'NDCG@{k}' for k in KS] + ['MRR'] + \
            [c for c in baselines.columns if c.startswith('Coverage')]
baselines[show_cols].round(4)

In [ ]:
avg_unmasked = N_ITEMS - (seq.n_train.mean() + 1)
for split in ['val', 'test']:
    r_full = baselines.query("model=='random' and protocol=='full' and split==@split").iloc[0]
    r_samp = baselines.query("model=='random' and protocol=='sampled' and split==@split").iloc[0]
    p_full = baselines.query("model=='popularity' and protocol=='full' and split==@split").iloc[0]
    exp_full, exp_samp = 10 / avg_unmasked, 10 / (data['neg_val'].shape[1] + 1)
    print(f'[{split}] random Recall@10 full={r_full["Recall@10"]:.5f} (expected ~{exp_full:.5f}) | '
          f'sampled={r_samp["Recall@10"]:.4f} (expected ~{exp_samp:.4f}) | popularity full={p_full["Recall@10"]:.4f}')
    assert abs(r_samp['Recall@10'] - exp_samp) < max(0.02, 4 * np.sqrt(exp_samp / len(seq))), 'random sampled metric off'
    assert r_full['Recall@10'] < 5 * exp_full + 0.01, 'random full-ranking metric too high: check masking'
    if p_full['Recall@10'] <= r_full['Recall@10']:
        print('  WARNING: popularity does not beat random; inspect the data.')
print('PASS random baseline matches theory')

### 4c. Group breakdown and diversity (popularity, test)
This is the format every model will report for the sparse-user and long-tail analysis.

In [ ]:
bd = R.group_breakdown(pop_test_res, data)
display(bd[['grouping', 'group', 'n_cases', 'Recall@10', 'NDCG@10', 'MRR']].round(4))
items_tbl = pd.read_parquet(PROCESSED / 'item_text.parquet')
cat_col = 'cat_l2' if dataset_info['subset']['kind'] == 'category' else 'cat_l1'
item_cat = np.array(['padding'] + items_tbl.sort_values('item_idx')[cat_col].fillna('unknown').tolist(), dtype=object)
print('Popularity category diversity@10:', round(R.category_diversity(pop_test_res['topk'][:, :10], item_cat), 4))

## 5. Cold-start and client files

In [ ]:
cold = pd.read_parquet(PROCESSED / 'cold_start' / 'cold_sequences.parquet')
cold_items = np.load(PROCESSED / 'cold_start' / 'cold_items.npy')
clients = pd.read_parquet(PROCESSED / 'clients' / 'clients.parquet')
assert len(clients) == data['n_users'] and clients.client_id.is_unique
assert cold_items.min() >= 1 and cold_items.max() <= N_ITEMS
print(f'Cold-start: {len(cold_items):,} cold items, {int(cold.test_is_cold.sum()):,} cold test cases')
print(f'Clients: {len(clients):,}; median local training size {clients.n_local_train.median():.0f}')
print(f'Item text: {len(items_tbl):,} rows; review snippets for {(items_tbl.n_review_snippets > 0).mean():.1%} of items')

## 6. Manifest (the freeze)
Every file in `processed/` with its checksum and size. If a teammate's checksums differ, they are not using the
frozen dataset.

In [ ]:
files = {}
for f in sorted(PROCESSED.rglob('*')):
    if f.is_file() and f.name not in ('manifest.json',) and '__pycache__' not in f.parts:
        files[str(f.relative_to(PROCESSED))] = {'sha256': sha256_file(f), 'bytes': f.stat().st_size}
source_manifest = json.loads((RAW / 'source_manifest.json').read_text())
manifest = {
    'created_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'dataset_info': dataset_info,
    'config': CFG,
    'source_files': source_manifest['files'],
    'processed_files': files,
    'versions': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__},
}
save_json(manifest, PROCESSED / 'manifest.json')
print(f'Manifest written with {len(files)} files')
pd.DataFrame(files).T.assign(MB=lambda d: (d.bytes / 1e6).round(2))[['MB', 'sha256']]

## 7. Data card

In [ ]:
stats = pd.read_csv(REPORTS / 'dataset_statistics.csv', index_col=0)
clean_log = json.loads((INTERIM / 'cleaning_log.json').read_text())
cold_info = json.loads((PROCESSED / 'cold_start' / 'cold_start_info.json').read_text())
client_stats = json.loads((PROCESSED / 'clients' / 'client_stats.json').read_text())
ug = data['user_groups'].user_group.value_counts().to_dict()
pop_row = baselines.query("model=='popularity' and split=='test' and protocol=='full'").iloc[0]

def md_table(df):
    cols = [str(c) for c in df.columns]
    lines = ['| ' + ' | '.join([''] + cols) + ' |', '|' + '---|' * (len(cols) + 1)]
    for idx, row in df.iterrows():
        lines.append('| ' + ' | '.join([str(idx)] + [str(v) for v in row.values]) + ' |')
    return '\n'.join(lines)

card = f"""# Data card: {CFG['dataset']} (frozen for Models 0-4)

Created: {manifest['created_utc']}

## Source
Amazon Reviews 2023 (Hou et al., 2024), McAuley Lab, UCSD. Official 5-core ratings + product metadata
{'+ review text (training interactions only)' if CFG['include_review_text'] else '(no review text)'}.

## Subset
{dataset_info['subset']['kind']} = **{dataset_info['subset']['value']}**. Reason: {dataset_info['subset']['reason']}

## Processing (in order)
1. Cleaning: missing values, ratings outside 1-5, timestamps outside 1996-2023, exact duplicates removed;
   repeat purchases of the same item -> earliest kept; items without a title dropped (shared item set for all models).
2. Subset applied, then iterative {CFG['kcore']}-core until stable.
3. IDs re-indexed: items 1..{dataset_info['n_items']:,} (0 = padding), users 0..{dataset_info['n_users'] - 1:,}.
4. Chronological leave-one-out split, ties broken by original file order. Max sequence length {CFG['max_seq_len']}.

## Statistics
{md_table(stats[['users', 'items', 'interactions', 'density_%', 'avg_seq_len', 'median_seq_len']])}

## Evaluation protocol (identical for all models)
- Main: full-catalog ranking, items in the user's history masked. Secondary: {data['neg_val'].shape[1]} uniform sampled negatives (fixed).
- Metrics: Recall@K, HitRate@K, NDCG@K (K = {', '.join(map(str, KS))}), MRR, Coverage@K, category diversity.
- Model selection on validation NDCG@10; test evaluated once. Report mean over seeds 42, 43, 44.
- Popularity baseline (test, full ranking): Recall@10 = {pop_row['Recall@10']:.4f}, NDCG@10 = {pop_row['NDCG@10']:.4f}.

## Analysis groups
- Users by training length: {ug}
- Items by training popularity: head = top {int(100 * CFG['item_pop']['head_frac'])}%, tail = bottom {int(100 * CFG['item_pop']['tail_frac'])}%.
- Cold start: {cold_info['n_cold_items']:,} new items held out of training; {cold_info['n_cold_test_cases']:,} cold test cases.

## Federated clients
One client per user ({client_stats['n_clients']:,} clients). Median local training size
{client_stats['local_train_size']['50%']}, size Gini {client_stats['local_size_gini']},
median KL(client || global) over {client_stats['category_level_for_non_iid']} = {client_stats['kl_to_global']['50%']}.

## Leakage safeguards (all passed)
{chr(10).join('- ' + k for k, v in leak['checks'].items())}
- Review snippets for item text come only from training interactions.

## Known limitations
- Leave-one-out uses a per-user cut, so some training interactions of one user occur after another user's test time
  (standard in sequential-recommendation benchmarks; the timestamp split is an alternative).
- BLaIR was pretrained on Amazon Reviews 2023; compare against a general text encoder to rule out leakage through it.
- Reviews are a proxy for purchases; ratings are kept only as side information.
"""
(PROCESSED / 'DATA_CARD.md').write_text(card)
print(card)

## Done: the dataset is frozen
Model notebooks should start with:
```python
import sys; sys.path.insert(0, str(PROCESSED)); import rec_eval as R
data = R.load_dataset(PROCESSED)
X_tr, Y_tr = R.train_arrays(data['sequences'], data['max_seq_len'])
res = R.evaluate(my_score_fn, data, split='val')   # select on validation NDCG@10
```
**Next stage:** Model 0 (SASRec baseline), then Model 1 (LLM profiles → embeddings → fusion → distillation).